# Iris Classification — Mage Pipeline

**Project:** `iris_project/iris_classifier`  
**Purpose:** Load and validate the Iris dataset, prepare features, train a decision-tree classifier, evaluate it, summarize the evaluation, and export predictions.

> This notebook is a companion evidence notebook reconstructed from the existing Mage project source and saved execution evidence. It is not represented as the original Mage UI notebook. Saved outputs below are transcribed from the project's evidence files; rerunning the Mage pipeline is not required to view them.

## 1. Pipeline architecture

The pipeline contains seven blocks. The training block branches into evaluation and result export; evaluation then feeds the custom summary block.

```text
load_iris
    |
validate_data
    |
prepare_data
    |
train_model
   /       \
evaluate   save_results
   |
summarize_evaluation
```

In [1]:
from pathlib import Path
import json
import csv

PROJECT = Path("iris_project")
EVIDENCE = Path("evidence")

print("Project directory exists:", PROJECT.exists())
print("Pipeline metadata exists:", (PROJECT / "pipelines/iris_classifier/metadata.yaml").exists())
print("Evidence directory exists:", EVIDENCE.exists())

Project directory exists: True
Pipeline metadata exists: True
Evidence directory exists: True


## 2. Mage block responsibilities

| Block | Type | Responsibility |
|---|---|---|
| `load_iris` | Data loader | Load the Iris dataset |
| `validate_data` | Transformer | Validate input data and expected structure |
| `prepare_data` | Transformer | Prepare features and target, then split data |
| `train_model` | Transformer | Train the decision-tree model; tested `max_depth` values 1–5 |
| `evaluate_model` | Transformer | Calculate classification metrics and confusion matrix |
| `summarize_evaluation` | Transformer | Produce a concise evaluation summary with a custom `@test` |
| `save_results` | Data exporter | Save predictions/results |

The saved pipeline metadata lists all seven blocks and their upstream/downstream relationships. Its status fields are `not_executed`; successful execution is supported separately by the captured run/evaluation evidence.

In [2]:
# Read the saved pipeline metadata (run from the repository root).
metadata_path = PROJECT / "pipelines/iris_classifier/metadata.yaml"
print(metadata_path.read_text(encoding="utf-8"))

blocks:
- name: load_iris
  type: data_loader
  downstream_blocks: [validate_data]
  upstream_blocks: []
- name: validate_data
  type: transformer
  downstream_blocks: [prepare_data]
  upstream_blocks: [load_iris]
- name: prepare_data
  type: transformer
  downstream_blocks: [train_model]
  upstream_blocks: [validate_data]
- name: train_model
  type: transformer
  downstream_blocks: [evaluate_model, save_results]
  upstream_blocks: [prepare_data]
- name: evaluate_model
  type: transformer
  downstream_blocks: [summarize_evaluation]
  upstream_blocks: [train_model]
- name: summarize_evaluation
  type: transformer
  downstream_blocks: []
  upstream_blocks: [evaluate_model]
- name: save_results
  type: data_exporter
  downstream_blocks: []
  upstream_blocks: [train_model]

Note: the full metadata file also contains UUIDs, widgets, and status fields.


## 3. Evaluation results

The following values are from `evidence/C3_evaluation_metrics.txt`. The evaluation block test was recorded as passed.

In [3]:
evaluation = {
    "labels": [0, 1, 2],
    "confusion_matrix": [[7, 0, 0], [0, 11, 0], [0, 1, 11]],
    "accuracy": 0.9666666667,
    "weighted_precision": 0.9694444444,
    "weighted_recall": 0.9666666667,
    "weighted_f1": 0.9666666667,
}
print(f"Accuracy: {evaluation['accuracy']:.4%}")
print(f"Weighted precision: {evaluation['weighted_precision']:.4%}")
print(f"Weighted recall: {evaluation['weighted_recall']:.4%}")
print(f"Weighted F1: {evaluation['weighted_f1']:.4%}")
print("Confusion matrix:")
for row in evaluation["confusion_matrix"]:
    print(row)
print("Mage evaluation block test: PASSED")

Accuracy: 96.6667%
Weighted precision: 96.9444%
Weighted recall: 96.6667%
Weighted F1: 96.6667%
Confusion matrix:
[7, 0, 0]
[0, 11, 0]
[0, 1, 11]
Mage evaluation block test: PASSED


## 4. Custom evaluation summary

The custom `summarize_evaluation` block reports accuracy, performance category, class count, and confusion matrix. Its captured output is below.

In [4]:
summary = {
    "accuracy": 0.9666666667,
    "performance": "excellent",
    "class_count": 3,
    "confusion_matrix": [
        [7, 0, 0],
        [0, 11, 0],
        [0, 1, 11],
    ],
}
print(json.dumps(summary, indent=2))

{
  "accuracy": 0.9666666667,
  "performance": "excellent",
  "class_count": 3,
  "confusion_matrix": [
    [7, 0, 0],
    [0, 11, 0],
    [0, 1, 11]
  ]
}


## 5. Decision-tree depth sweep

The saved sweep (`evidence/C2_max_depth_sweep_verified.csv`) evaluates `max_depth` values 1 through 5. Each run recorded 29 correct predictions out of 30 (96.67%) and 6/6 Mage tests passed. The additional `test_accuracy_range_check` column records the accuracy-range assertion as passing.

In [5]:
sweep_path = EVIDENCE / "C2_max_depth_sweep_verified.csv"
with sweep_path.open(newline="", encoding="utf-8") as f:
    sweep_rows = list(csv.DictReader(f))

headers = ["max_depth", "correct", "total", "accuracy", "mage_tests", "test_accuracy_range_check"]
print(" | ".join(headers))
print("-" * 88)
for row in sweep_rows:
    print(" | ".join(row.get(h, "") for h in headers))

max_depth | correct | total | accuracy | mage_tests | test_accuracy_range_check
----------------------------------------------------------------------------------------
1 | 29 | 30 | 96.67% | 6/6 passed | PASS
2 | 29 | 30 | 96.67% | 6/6 passed | PASS
3 | 29 | 30 | 96.67% | 6/6 passed | PASS
4 | 29 | 30 | 96.67% | 6/6 passed | PASS
5 | 29 | 30 | 96.67% | 6/6 passed | PASS


## 6. Prediction export evidence

The project also saved `iris_predictions.csv` in the repository root. The recorded validation contained 30 test rows, with 29 correct predictions and accuracy of 0.9667 (96.67%). The exporter block is `save_results`, downstream of `train_model`.

In [6]:
predictions_path = Path("iris_predictions.csv")
print("Prediction CSV exists:", predictions_path.exists())
if predictions_path.exists():
    with predictions_path.open(newline="", encoding="utf-8") as f:
        prediction_rows = list(csv.DictReader(f))
    print("Prediction rows:", len(prediction_rows))
print("Recorded correct predictions: 29 / 30")
print("Recorded test accuracy: 0.9667")

Prediction CSV exists: True
Prediction rows: 30
Recorded correct predictions: 29 / 30
Recorded test accuracy: 0.9667


## 7. Evidence references

The following files are retained in the repository's `evidence/` folder:

- `C1_mage_pipeline_graph.png` — Mage pipeline graph
- `C2_max_depth_sweep.csv` and `C2_max_depth_sweep_verified.csv` — depth sweep results
- `C3_evaluation_metrics.txt` — evaluation metrics and confusion matrix
- `C3_metadata.yaml` — copied pipeline metadata
- `C4_summary_output.json` — custom evaluation summary output

## 8. Conclusion

The saved run evidence records a seven-block Iris classification pipeline, passing block tests, an evaluation accuracy of **96.67%**, and a custom summary categorized as **excellent**. All five tested tree depths achieved the same recorded accuracy on the 30-row test set. The notebook preserves the saved results for review and distinguishes those results from the metadata's static `not_executed` status fields.